# Text Analytics Coursework

This notebook provides some example code for loading and examining the dataset for task 2. 

hello


In [ ]:
#%load_ext autoreload
#%autoreload 2



In [56]:
# Use HuggingFace's datasets library to access the Emotion dataset
from datasets import load_dataset
import numpy as np
import pandas as pd

# Task 2 - EBM-NLP

This dataset is provided at https://github.com/bepnye/EBM-NLP and a copy has been made available in this repository for convenience. The data will need to be unzipped:

In [57]:
import tarfile
import os

path_tofile = "C:/Users/Hp/Downloads/EBM-NLP/ebm_nlp_1_00.tar.gz"
extract_directory = os.path.dirname(path_tofile)

if tarfile.is_tarfile(path_tofile):
    with tarfile.open(path_tofile) as f:
        f.extractall(path=extract_directory)  # Extract all members from the archive to the current working directory


The data contains text documents that are annotated for mentions of participants, interventions and outcomes (PIO) in medical research. For each entity type, P, I, or O, there is a slightly different set of documents in the training and test set. Most of the documents are identical, but each type has a few extra documents. So, let's deal with each type separately for now.

To load the text documents, we first make a list of the document IDs for one entity type (P, I or O):

In [ ]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_1_00")

print("Exists:", DATA_DIR.exists())
print("Contents:", list(DATA_DIR.iterdir()))


Current working directory: c:\Users\Hp\Downloads\EBM-NLP\notebooks
Files/folders here:
['.venv', 'draft.ipynb', 'task2_ebm-nlp edited.ipynb', 'task2_ebm-nlp.ipynb']
.venv
draft.ipynb
task2_ebm-nlp edited.ipynb
task2_ebm-nlp.ipynb


Now, we can get the annotations for the first entity type:

In [58]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_1_00")

train_dir = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / "participants" / "train"
test_dir = DATA_DIR / "annotations" / "aggregated" / "hierarchical_labels" / "participants" / "test" / "gold"


train_files = list(train_dir.glob("*_AGGREGATED.ann"))
print("Number of matching train files:", len(train_files))
print("First 5 train files:", [p.name for p in train_files[:5]])

train_doc_ids = [p.name.replace("_AGGREGATED.ann", "") for p in train_files]
print("First 5 extracted train doc ids:", train_doc_ids[:5])

test_files = list(test_dir.glob("*_AGGREGATED.ann"))
print("Number of matching test files:", len(test_files))
print("First 5 test files:", [p.name for p in test_files[:5]])

test_doc_ids = [p.name.replace("_AGGREGATED.ann", "") for p in test_files]
print("First 5 extracted test doc ids:", test_doc_ids[:5])

Number of matching train files: 3487
First 5 train files: ['10036953_AGGREGATED.ann', '10052279_AGGREGATED.ann', '10070173_AGGREGATED.ann', '10075386_AGGREGATED.ann', '10077140_AGGREGATED.ann']
First 5 extracted train doc ids: ['10036953', '10052279', '10070173', '10075386', '10077140']
Number of matching test files: 200
First 5 test files: ['10084579_AGGREGATED.ann', '10674229_AGGREGATED.ann', '10690697_AGGREGATED.ann', '10707032_AGGREGATED.ann', '10715372_AGGREGATED.ann']
First 5 extracted test doc ids: ['10084579', '10674229', '10690697', '10707032', '10715372']


In [59]:
from pathlib import Path

DATA_DIR = Path("./ebm_nlp_1_00")

def get_doc_ids(split="train", label_type="participants"):
    if split == "test":
        label_dir = (
            DATA_DIR
            / "annotations"
            / "aggregated"
            / "hierarchical_labels"
            / label_type
            / "test"
            / "gold"
        )
    else:
        label_dir = (
            DATA_DIR
            / "annotations"
            / "aggregated"
            / "hierarchical_labels"
            / label_type
            / "train"
        )

    print("Looking in:", label_dir)
    print("Exists:", label_dir.exists())

    doc_files = list(label_dir.glob("*_AGGREGATED.ann"))
    doc_ids = [p.name.replace("_AGGREGATED.ann", "") for p in doc_files]

    return sorted(doc_ids)

doc_ids_p = get_doc_ids("train", "participants")
test_doc_ids_p = get_doc_ids("test", "participants")

print(f"Number of documents in train split for participants: {len(doc_ids_p)}")
print(f"Number of documents in test split for participants: {len(test_doc_ids_p)}")
print("First 5 train IDs:", doc_ids_p[:5])

Looking in: ebm_nlp_1_00\annotations\aggregated\hierarchical_labels\participants\train
Exists: True
Looking in: ebm_nlp_1_00\annotations\aggregated\hierarchical_labels\participants\test\gold
Exists: True
Number of documents in train split for participants: 3487
Number of documents in test split for participants: 200
First 5 train IDs: ['10036953', '10052279', '10070173', '10075386', '10077140']


In [60]:
print("len(doc_ids_p):", len(doc_ids_p))
print("len(participants_labels):", len(participants_labels))
sample = min(5, len(participants_labels) - 1)
print("Document ID:", doc_ids_p[sample])
print(f"Participants label example for doc {doc_ids_p[sample]}:")
print(participants_labels[sample])


len(doc_ids_p): 3487
len(participants_labels): 3487
Document ID: 10078672
Participants label example for doc 10078672:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 

In [61]:

def load_labels_for_doc(doc_id, label_type="participants", split="train"):
    """
    label_type: 'participants', 'interventions', or 'outcomes'
    split: 'train' or 'test'
    """
    if split == "test":
        split = "test/gold"

    ann_path = (
        DATA_DIR
        / "annotations"
        / "aggregated"
        / "hierarchical_labels"
        / label_type
        / split
        / f"{doc_id}_AGGREGATED.ann"
    )

    if not ann_path.exists():
        print(ann_path, "does not exist!")
        return None

    with open(ann_path, "r", encoding="utf-8") as f:
        line = f.readline().strip()
        labels = line.split(",")

    return labels


def load_labels(doc_ids, label_type="participants", split="train"):
    labels = []
    for doc_id in doc_ids:
        doc_labels = load_labels_for_doc(doc_id, label_type, split)
        if doc_labels is not None:
            labels.append(doc_labels)
    return labels

    return labels

participants_labels = load_labels(doc_ids_p, "participants", split="train")

print(f"Length of participants_labels: {len(participants_labels)}")

test_participants_labels = load_labels(test_doc_ids_p, "participants", split="test")
print(f"Length of test_participants_labels: {len(test_participants_labels)}")

sample = 123
print("Document ID:", doc_ids_p[sample])
print(f"Participants label example for doc {doc_ids_p[sample]}:")
print(participants_labels[sample])

Length of participants_labels: 3487
Length of test_participants_labels: 200
Document ID: 10703628
Participants label example for doc 10703628:
['0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '4', '4', '4', '4', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '3', '3', '0', '3', '3', '0', '0', '0', '0', '0', '4', '4', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '0', '

In [62]:
from itertools import chain
import numpy as np

all_labels = chain(*participants_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['0' '1' '2' '3' '4']


Let's look at what labels there are for Participants. The code above shows there are four values: 0 corresponds to 'outside' but 1-4 all indicate tokens that form an entity span. Each number is a level in a hierarchy of specificity. To start with let's not worry about this 'hierarchy'. We can instead just turn the labels into simple BIO (Beginning of a span, Inside a span, and Outside a span) tags.

In [63]:
from itertools import chain
def hierarchical_to_bio(tags):
    """
    Convert EBM-NLP hierarchical labels (0–4) to flat BIO tags.

    Parameters
    ----------
    tags : list[int]
        A list of hierarchical labels for a single document.

    Returns
    -------
    list[str]
        BIO tags ("O", "B", "I").
    """

    bio = []
    prev = 0

    for t in tags:
        t = int(t)  # ensure it's an integer
        if t == 0:
            bio.append("O")
        else:
            if prev == 0:
                bio.append("B")
            else:
                bio.append("I")

        prev = t
        

    return bio

def convert_all_labels_to_bio(labels):
    for i, doc_labels in enumerate(labels):
        labels[i] = hierarchical_to_bio(doc_labels)
    return labels

participants_labels = convert_all_labels_to_bio(participants_labels)
test_participants_labels = convert_all_labels_to_bio(test_participants_labels)

all_labels = chain(*participants_labels)

# show how many unique labels there are across the dataset
print(np.unique(list(all_labels)))

['B' 'I' 'O']


So far, we've loaded the document IDs for participants and the corresponding labels. Now, let's load the documents themselves. They are already tokenised so that the labels match up with the tokens:

In [64]:
print(len(doc_ids_p))

3487


In [65]:
def load_document(doc_id):
    doc_path = DATA_DIR / "documents" / f"{doc_id}.tokens"
    with open(doc_path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

def load_documents(doc_ids):
    documents = []
    for doc_id in doc_ids:
        doc = load_document(doc_id)
        documents.append(doc)
    return documents

participants_tokens = load_documents(doc_ids_p)
test_participants_tokens = load_documents(test_doc_ids_p)

# inspect a random element
sample = min(5, len(doc_ids_p) - 1)
print("Document ID:", doc_ids_p[sample])
print(f"Tokenised document example for doc {doc_ids_p[sample]}:")
print(participants_tokens[sample])


Document ID: 10078672
Tokenised document example for doc 10078672:
['Behavioral and physiological effects of remifentanil and alfentanil in healthy volunteers . BACKGROUND The subjective and psychomotor effects of remifentanil have not been evaluated . Accordingly , the authors used mood inventories and psychomotor tests to characterize the effects of remifentanil in healthy , non-drug-abusing volunteers . Alfentanil was used as a comparator drug . METHODS Ten healthy volunteers were enrolled in a randomized , double-blinded , placebo-controlled , crossover trial in which they received an infusion of saline , remifentanil , or alfentanil for 120 min . The age- and weight-adjusted infusions ( determined with STANPUMP , a computer modeling software package ) were given to achieve three predicted constant plasma levels for 40 min each of remifentanil ( 0.75 , 1.5 , and 3 ng/ml ) and alfentanil ( 16 , 32 , and 64 ng/ml ) . Mood forms and psychomotor tests were completed , and miosis was as

### Interventions

In [66]:
doc_ids_i = get_doc_ids("train", "interventions")
test_doc_ids_i = get_doc_ids("test", "interventions")

print(f"Number of documents in train split for interventions: {len(doc_ids_i)}")
print(f"Number of documents in test split for interventions: {len(test_doc_ids_i)}")

interventions_tokens = load_documents(doc_ids_i)
test_interventions_tokens = load_documents(test_doc_ids_i)

Looking in: ebm_nlp_1_00\annotations\aggregated\hierarchical_labels\interventions\train
Exists: True
Looking in: ebm_nlp_1_00\annotations\aggregated\hierarchical_labels\interventions\test\gold
Exists: True
Number of documents in train split for interventions: 3314
Number of documents in test split for interventions: 200


In [67]:
interventions_labels = load_labels(doc_ids_i, "interventions", split="train")
print(f"Length of interventions_labels: {len(interventions_labels)}")

test_interventions_labels = load_labels(test_doc_ids_i, "interventions", split="test")
print(f"Length of test_interventions_labels: {len(test_interventions_labels)}")

interventions_labels = convert_all_labels_to_bio(interventions_labels)
test_interventions_labels = convert_all_labels_to_bio(test_interventions_labels)

Length of interventions_labels: 3314
Length of test_interventions_labels: 200


### Outcomes

In [68]:
doc_ids_o = get_doc_ids("train", "outcomes")
test_doc_ids_o = get_doc_ids("test", "outcomes")

print(f"Number of documents in train split for outcomes: {len(doc_ids_o)}")
print(f"Number of documents in test split for outcomes: {len(test_doc_ids_o)}")

outcomes_tokens = load_documents(doc_ids_o)
test_outcomes_tokens = load_documents(test_doc_ids_o)

Looking in: ebm_nlp_1_00\annotations\aggregated\hierarchical_labels\outcomes\train
Exists: True
Looking in: ebm_nlp_1_00\annotations\aggregated\hierarchical_labels\outcomes\test\gold
Exists: True
Number of documents in train split for outcomes: 3876
Number of documents in test split for outcomes: 200


In [69]:
outcomes_labels = load_labels(doc_ids_o, "outcomes", split="train")
print(f"Length of outcomes_labels: {len(outcomes_labels)}")

test_outcomes_labels = load_labels(test_doc_ids_o, "outcomes", split="test")   
print(f"Length of test_outcomes_labels: {len(test_outcomes_labels)}")

outcomes_labels = convert_all_labels_to_bio(outcomes_labels)
test_outcomes_labels = convert_all_labels_to_bio(test_outcomes_labels)


Length of outcomes_labels: 3876
Length of test_outcomes_labels: 200


In [70]:
print(f"Intersection of train doc IDs across entity types: {len(set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o))}")
print(f"Intersection of test doc IDs across entity types: {len(set(test_doc_ids_p) & set(test_doc_ids_i) & set(test_doc_ids_o))}")
print(f"Documents that are different across entity types in train split: {len((set(doc_ids_p) | set(doc_ids_i) | set(doc_ids_o)) - (set(doc_ids_p) & set(doc_ids_i) & set(doc_ids_o)))}")
print(f"Documents that are different across entity types in test split: {len((set(test_doc_ids_p) | set(test_doc_ids_i) | set(test_doc_ids_o)) - (set(test_doc_ids_p) & set(test_doc_ids_i) & set(test_doc_ids_o)))}")


print(f"Test examples of the participants type that are in other entity types' training splits: {set(test_doc_ids_p) & (set(doc_ids_i) | set(doc_ids_o))}")
print(f"Test examples of the interventions type that are in other entity types' training splits: {set(test_doc_ids_i) & (set(doc_ids_p) | set(doc_ids_o))}")
print(f"Test examples of the outcomes type that are in other entity types' training splits: {set(test_doc_ids_o) & (set(doc_ids_p) | set(doc_ids_i))}")

Intersection of train doc IDs across entity types: 2782
Intersection of test doc IDs across entity types: 99
Documents that are different across entity types in train split: 1263
Documents that are different across entity types in test split: 202
Test examples of the participants type that are in other entity types' training splits: {'17662102', '16382035', '16731878', '9564194', '11501687', '16167251', '23280086', '16095446', '22810989', '23326865', '16304214', '22565161', '20830241', '20345030', '19376304', '16199793', '22048089', '18795522', '22982948', '16293958', '19081412', '15897310', '17477785', '16603337', '17617281', '21439528', '23257173', '22219012', '20875504', '18544974', '8126502', '16116055', '11100343', '24201232', '1878735', '19968217', '21721430', '17156222', '18397386', '19073386', '23296213', '22525955', '23103798', '18565251', '19462303', '16257339', '20633669', '16804044', '7831628', '22395144', '17152183', '16791814', '11041498', '18501909', '16275518', '2049443

In [71]:
sample_doc = participants_tokens[0]
print(type(sample_doc))
print("length:", len(sample_doc))
print(sample_doc[:20])

<class 'list'>
length: 1
['[ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] . Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection . To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection , 99 patients with H. pylori infection were treated with either lansoprazole ( LPZ ) or ranitidine ( RNT ) used together with clarithromycin ( CAM ) and metrinidazole ( MTZ ) . The drug combination and administration periods in the PPI group were LPZ 30 mg , CAM 400 mg , MTZ 500 mg ( LCM group ) . The ranitidine group received RNT 300 mg , CAM 400 mg , MTZ 500 mg ( RCM group ) . The cure rate of H. pylori infection was 88 % in the LCM group ; 95 % CI 79-97 and 92 % in the RCM group ; 95 % CI 84-99 .']


In [72]:
#clustering on sentence embeddings shannon task

#Load document
train_doc_ids=doc_ids_p
tokenised_abstracts=participants_tokens

print("number of abstracts:", len(tokenised_abstracts))
print("example token:",tokenised_abstracts[0][:30])




number of abstracts: 3487
example token: ['[ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] . Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection . To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection , 99 patients with H. pylori infection were treated with either lansoprazole ( LPZ ) or ranitidine ( RNT ) used together with clarithromycin ( CAM ) and metrinidazole ( MTZ ) . The drug combination and administration periods in the PPI group were LPZ 30 mg , CAM 400 mg , MTZ 500 mg ( LCM group ) . The ranitidine group received RNT 300 mg , CAM 400 mg , MTZ 500 mg ( RCM group ) . The cure rate of H. pylori infection was 88 % in the LCM group ; 95 % CI 79-97 and 92 % in the RCM group ; 95 % CI 84-99 .']


In [73]:
#Tokens to full text abstract conversion

abstract_texts = [tokens[0] for tokens in tokenised_abstracts]
print("example abstract:")
print(abstract_texts[0][:500])


example abstract:
[ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] . Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection . To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection , 99 patients with H. pylori infection were treated with either lansoprazole ( LPZ ) or ranitidine ( RNT ) used together with clarit


In [74]:
print(sentence_df["cluster_id"].value_counts())

cluster_id
2    10701
1    10385
3     9986
0     8446
Name: count, dtype: int64


In [75]:
#splitting abstracts to sentences
import nltk
nltk.download("punkt")

from nltk.tokenize import sent_tokenize
all_sentences=[]
sentence_source_doc=[]

#break each abstract 
for doc_id, abstract in zip(train_doc_ids, abstract_texts):
    sentences=sent_tokenize(abstract)

    for sentence in sentences:
        if sentence.strip():
          all_sentences.append(sentence)
          sentence_source_doc.append(doc_id)

print("total number of sentences:", len(all_sentences))
print("\nExample sentences:")
for s in all_sentences[:5]:
   print("-",s)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


total number of sentences: 39518

Example sentences:
- [ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] .
- Comparison of ranitidine and lansoprazole in short-term low-dose triple therapy for Helicobacter pylori infection .
- To evaluate the efficacy and safety of two 1-week low-dose triple-therapy drug regimens involving antisecretory drugs for Helicobacter pylori infection , 99 patients with H. pylori infection were treated with either lansoprazole ( LPZ ) or ranitidine ( RNT ) used together with clarithromycin ( CAM ) and metrinidazole ( MTZ ) .
- The drug combination and administration periods in the PPI group were LPZ 30 mg , CAM 400 mg , MTZ 500 mg ( LCM group ) .
- The ranitidine group received RNT 300 mg , CAM 400 mg , MTZ 500 mg ( RCM group ) .


In [76]:
#sentenceembeddings
from sentence_transformers import SentenceTransformer
embedding_model=SentenceTransformer("all-MiniLM-L6-v2")
sentence_vector=embedding_model.encode(all_sentences, show_progress_bar=True)
print("shape of sentence vector", sentence_vector.shape)



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 865.15it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1235/1235 [16:58<00:00,  1.21it/s]


shape of sentence vector (39518, 384)


In [77]:

#kmeans cluster
from sklearn.cluster import KMeans

#k=4
num_cluster=4
kmeans_model=KMeans(n_clusters=num_cluster, random_state=42)
cluster_assign=kmeans_model.fit_predict(sentence_vector)
print("first few cluster assignments")
print(cluster_assign[:20])


first few cluster assignments
[3 3 3 3 3 2 2 2 2 1 1 2 1 1 2 2 3 0 3 0]


In [78]:
#results in dataframe
import pandas as pd

sentence_df=pd.DataFrame({
    "document_id":sentence_source_doc,
    "sentence_text":all_sentences,
    "cluster_id":cluster_assign
})
sentence_df.head()


,document_id,sentence_text,cluster_id
0,10036953,[ Triple therapy regimens involving H2 blockad...,3
1,10036953,Comparison of ranitidine and lansoprazole in s...,3
2,10036953,To evaluate the efficacy and safety of two 1-w...,3
3,10036953,The drug combination and administration period...,3
4,10036953,"The ranitidine group received RNT 300 mg , CAM...",3


In [79]:
#evaluate
from sklearn.metrics import silhouette_score
silhouette=silhouette_score(sentence_vector,cluster_assign)
print("silhouette score",silhouette)

silhouette score 0.02299085445702076


In [80]:
for cluster_id in sorted(sentence_df["cluster_id"].unique()):
    print(f"\n===== CLUSTER{cluster_id} =====")

    example_sentences=sentence_df[sentence_df["cluster_id"]== cluster_id]["sentence_text"].head(10)
    for e in example_sentences:
        print("-", e)


===== CLUSTER0 =====
- Rhinocort Study Group .
- Secondarily to ascertain patients ' preferences for the two nasal devices and to assess quality of life .
- DESIGN Randomized , multicentre , double-blind , double- dummy , parallel groups study .
- Improvement in quality of life from baseline to clinic visits was statistically significant in both groups .
- According to a crossover design , they were randomized to have either sleep deprivation or a full night 's sleep 1 week apart , during which they were monitored with ABPM .
- Masticatory performance and chewing experience with implant-retained mandibular overdentures .
- The relationship between masticatory performance and chewing experience has not yet been explored for patients with implant-retained overdentures .
- Although many relationships have been found between parameters of objective and subjective oral function , the structure of these relationships remain unclear .
- Therefore , we studied in a randomized clinical trial t

In [81]:
for k in [3, 4, 5, 6]:
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(sentence_vector)
    score = silhouette_score(sentence_vector, labels)
    print(f"K={k}, silhouette={score}")

K=3, silhouette=0.02419559843838215
K=4, silhouette=0.02299085445702076
K=5, silhouette=0.022774798795580864


KeyboardInterrupt: 